In [1]:
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

REQUIRED_PACKAGES = ["einops"]

for package_name in REQUIRED_PACKAGES:
    if importlib.util.find_spec(package_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from monai.inferers import sliding_window_inference
from monai.networks.nets import AttentionUnet, DynUNet, SwinUNETR, UNet, ViT
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
plt.rcParams["figure.figsize"] = (12, 5)


2026-03-20 14:31:40.243956: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 14:31:40.296700: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-20 14:31:41.585638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
SEED = 42
IMAGE_SIZE = 512
PATCH_SIZE = 256
NUM_CLASSES = 3
TEST_RATIO = 0.2
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 4
NUM_EPOCHS = 40
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
PATIENCE = 8
SLIDING_WINDOW_BATCH = 4
SLIDING_WINDOW_OVERLAP = 0.25
FORCE_REPROCESS = False

DATA_ROOT = Path("./data")
X_PATH = Path("./X.npy")
Y_PATH = Path("./Y.npy")
SAMPLE_IDS_PATH = Path("./sample_ids.npy")
MANIFEST_PATH = Path("./sample_manifest.csv")

RUN_ROOT = Path("./runs/unet_virant")
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
PRED_DIR = RUN_ROOT / "predictions"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = [
    "unet",
    "attention_unet",
    "transunet",
    "swin_unet",
    "nnunet_style",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = torch.cuda.is_available()

set_seed(SEED)
print("Device:", DEVICE)


Device: cuda


In [3]:
def collect_samples(data_root: Path):
    samples = []
    for sample_dir in sorted(data_root.glob("*/*")):
        if not sample_dir.is_dir():
            continue
        image_candidates = sorted((sample_dir / "images").glob("*_mSLIM.png"))
        mask_candidates = sorted((sample_dir / "masks").glob("*.png"))
        if image_candidates and mask_candidates:
            samples.append(
                {
                    "group_id": sample_dir.parent.name,
                    "sample_id": sample_dir.name,
                    "sample_dir": sample_dir,
                    "image_path": image_candidates[0],
                    "mask_paths": mask_candidates,
                }
            )
    return samples


def preprocess_dataset(
    data_root: Path,
    image_size: int = IMAGE_SIZE,
    force: bool = False,
):
    if X_PATH.exists() and Y_PATH.exists() and MANIFEST_PATH.exists() and not force:
        x = np.load(X_PATH)
        y = np.load(Y_PATH)
        manifest = pd.read_csv(MANIFEST_PATH)
        print("Loaded cached arrays:", x.shape, y.shape)
        return x, y, manifest

    samples = collect_samples(data_root)
    images = []
    masks = []
    rows = []

    for sample in tqdm(samples, desc="Preprocessing raw samples"):
        image = np.array(Image.open(sample["image_path"]), dtype=np.float32)
        image = resize(
            image,
            (image_size, image_size),
            order=1,
            mode="constant",
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)

        low, high = np.percentile(image, (1, 99))
        if high > low:
            image = np.clip((image - low) / (high - low), 0.0, 1.0)
        else:
            image = np.zeros_like(image, dtype=np.float32)

        mask = np.zeros((image_size, image_size), dtype=np.uint8)
        for mask_path in sample["mask_paths"]:
            mask_arr = np.array(Image.open(mask_path), dtype=np.uint8)
            mask_arr = resize(
                mask_arr,
                (image_size, image_size),
                order=0,
                mode="constant",
                preserve_range=True,
                anti_aliasing=False,
            ).astype(np.uint8)
            class_value = 2 if "Prot" in mask_path.name else 1
            mask = np.maximum(mask, np.where(mask_arr > 0, class_value, 0).astype(np.uint8))

        images.append(image)
        masks.append(mask)
        rows.append(
            {
                "group_id": sample["group_id"],
                "sample_id": sample["sample_id"],
                "image_path": str(sample["image_path"]),
            }
        )

    x = np.stack(images, axis=0).astype(np.float32)
    y = np.stack(masks, axis=0).astype(np.uint8)
    manifest = pd.DataFrame(rows)

    np.save(X_PATH, x)
    np.save(Y_PATH, y)
    np.save(SAMPLE_IDS_PATH, manifest["sample_id"].to_numpy(dtype=object))
    manifest.to_csv(MANIFEST_PATH, index=False)

    print("Saved arrays to disk:", X_PATH, Y_PATH)
    print("X:", x.shape, x.dtype, "Y:", y.shape, y.dtype, "classes:", np.unique(y))
    return x, y, manifest


X, Y, manifest = preprocess_dataset(DATA_ROOT, force=FORCE_REPROCESS)
manifest.head()


Loaded cached arrays: (458, 512, 512) (458, 512, 512)


,group_id,sample_id,image_path
0,51,f0_t0_i0_ch0_c0_r0_z0,data/051/f0_t0_i0_ch0_c0_r0_z0/images/f0_t0_i0...
1,51,f0_t0_i10_ch0_c0_r0_z0,data/051/f0_t0_i10_ch0_c0_r0_z0/images/f0_t0_i...
2,51,f0_t0_i11_ch0_c0_r0_z0,data/051/f0_t0_i11_ch0_c0_r0_z0/images/f0_t0_i...
3,51,f0_t0_i12_ch0_c0_r0_z0,data/051/f0_t0_i12_ch0_c0_r0_z0/images/f0_t0_i...
4,51,f0_t0_i13_ch0_c0_r0_z0,data/051/f0_t0_i13_ch0_c0_r0_z0/images/f0_t0_i...


In [ ]:
def show_mask(mask, title=None):
    cmap = plt.cm.colors.ListedColormap(["black", "green", "yellow"])
    bounds = [-0.5, 0.5, 1.5, 2.5]
    norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)
    plt.imshow(mask, cmap=cmap, norm=norm)
    cbar = plt.colorbar(ticks=[0, 1, 2], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["Background", "Live", "Dead"])
    if title:
        plt.title(title)
    plt.axis("off")


idx = random.randint(0, len(X) - 1)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(X[idx], cmap="gray")
plt.title(f"Image {idx}")
plt.axis("off")

plt.subplot(1, 2, 2)
show_mask(Y[idx], title=f"Mask {idx}")
plt.tight_layout()
plt.show()


In [5]:
dead_present = (Y.reshape(len(Y), -1) == 2).any(axis=1).astype(np.int64)
indices = np.arange(len(X))

train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_RATIO,
    random_state=SEED,
    shuffle=True,
    stratify=dead_present,
)

np.save(RUN_ROOT / "train_indices.npy", train_idx)
np.save(RUN_ROOT / "test_indices.npy", test_idx)

class_counts = np.bincount(Y[train_idx].reshape(-1), minlength=NUM_CLASSES)
class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print("Train/Test:", len(train_idx), len(test_idx))
print("Train class counts:", class_counts)
print("Class weights:", class_weights)


class CellSegmentationDataset(Dataset):
    def __init__(self, images, masks, indices, patch_size=None, train=False):
        self.images = images
        self.masks = masks
        self.indices = np.array(indices)
        self.patch_size = patch_size
        self.train = train

    def __len__(self):
        return len(self.indices)

    def random_crop(self, image, mask):
        if self.patch_size is None or image.shape[0] <= self.patch_size:
            return image, mask
        top = np.random.randint(0, image.shape[0] - self.patch_size + 1)
        left = np.random.randint(0, image.shape[1] - self.patch_size + 1)
        return (
            image[top : top + self.patch_size, left : left + self.patch_size],
            mask[top : top + self.patch_size, left : left + self.patch_size],
        )

    def augment(self, image, mask):
        if random.random() < 0.5:
            image = np.flip(image, axis=1).copy()
            mask = np.flip(mask, axis=1).copy()
        if random.random() < 0.5:
            image = np.flip(image, axis=0).copy()
            mask = np.flip(mask, axis=0).copy()
        rotations = np.random.randint(0, 4)
        if rotations:
            image = np.rot90(image, rotations).copy()
            mask = np.rot90(mask, rotations).copy()
        return image, mask

    def __getitem__(self, item):
        sample_idx = self.indices[item]
        image = self.images[sample_idx]
        mask = self.masks[sample_idx]

        if self.train:
            image, mask = self.random_crop(image, mask)
            image, mask = self.augment(image, mask)

        image_tensor = torch.from_numpy(image[None]).float()
        mask_tensor = torch.from_numpy(mask).long()
        return image_tensor, mask_tensor


train_dataset = CellSegmentationDataset(X, Y, train_idx, patch_size=PATCH_SIZE, train=True)
test_dataset = CellSegmentationDataset(X, Y, test_idx, patch_size=None, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=AMP_ENABLED,
    drop_last=False,
)


Train/Test: 366 92
Train class counts: [94552022   612678   780004]
Class weights: [0.01084812 1.67414415 1.31500773]


In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = ConvBlock(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class TransUNet2D(nn.Module):
    def __init__(
        self,
        img_size=PATCH_SIZE,
        in_channels=1,
        out_channels=NUM_CLASSES,
        hidden_size=192,
        mlp_dim=768,
        num_heads=3,
        num_layers=8,
        patch_size=16,
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size

        self.enc1 = ConvBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.vit = ViT(
            in_channels=in_channels,
            img_size=(img_size, img_size),
            patch_size=(patch_size, patch_size),
            hidden_size=hidden_size,
            mlp_dim=mlp_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            classification=False,
            spatial_dims=2,
        )
        self.vit_proj = nn.Sequential(
            nn.Conv2d(hidden_size, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.GELU(),
        )
        self.bridge = ConvBlock(512, 512)

        self.dec1 = DecoderBlock(512, 512, 256)
        self.dec2 = DecoderBlock(256, 256, 128)
        self.dec3 = DecoderBlock(128, 128, 64)
        self.dec4 = DecoderBlock(64, 64, 64)
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool1(s1))
        s3 = self.enc3(self.pool2(s2))
        s4 = self.enc4(self.pool3(s3))

        vit_tokens, _ = self.vit(x)
        bsz = vit_tokens.shape[0]
        bottleneck = vit_tokens.transpose(1, 2).reshape(bsz, -1, self.grid_size, self.grid_size)
        bottleneck = self.vit_proj(bottleneck)
        bottleneck = self.bridge(bottleneck)

        x = self.dec1(bottleneck, s4)
        x = self.dec2(x, s3)
        x = self.dec3(x, s2)
        x = self.dec4(x, s1)
        return self.out_conv(x)


def build_model(model_name: str):
    if model_name == "unet":
        return UNet(
            spatial_dims=2,
            in_channels=1,
            out_channels=NUM_CLASSES,
            channels=(32, 64, 128, 256, 512),
            strides=(2, 2, 2, 2),
            num_res_units=2,
        )

    if model_name == "attention_unet":
        return AttentionUnet(
            spatial_dims=2,
            in_channels=1,
            out_channels=NUM_CLASSES,
            channels=(32, 64, 128, 256, 512),
            strides=(2, 2, 2, 2),
            dropout=0.1,
        )

    if model_name == "transunet":
        return TransUNet2D(
            img_size=PATCH_SIZE,
            in_channels=1,
            out_channels=NUM_CLASSES,
            hidden_size=192,
            mlp_dim=768,
            num_heads=3,
            num_layers=8,
            patch_size=16,
        )

    if model_name == "swin_unet":
        return SwinUNETR(
            img_size=(PATCH_SIZE, PATCH_SIZE),
            in_channels=1,
            out_channels=NUM_CLASSES,
            feature_size=24,
            spatial_dims=2,
            use_checkpoint=False,
        )

    if model_name == "nnunet_style":
        return DynUNet(
            spatial_dims=2,
            in_channels=1,
            out_channels=NUM_CLASSES,
            kernel_size=[3, 3, 3, 3, 3],
            strides=[1, 2, 2, 2, 2],
            upsample_kernel_size=[2, 2, 2, 2],
            filters=[32, 64, 128, 256, 512],
            deep_supervision=False,
            res_block=True,
        )

    raise ValueError(f"Unknown model: {model_name}")


smoke_batch = torch.randn(1, 1, PATCH_SIZE, PATCH_SIZE)
for name in MODEL_NAMES:
    model = build_model(name)
    with torch.no_grad():
        out = model(smoke_batch)
    print(name, tuple(out.shape))


unet (1, 3, 256, 256)
attention_unet (1, 3, 256, 256)
transunet (1, 3, 256, 256)
swin_unet (1, 3, 256, 256)


/home/anzhe/anaconda3/envs/anzhe/lib/python3.10/site-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.swin_unetr SwinUNETR.__init__:img_size: Argument `img_size` has been deprecated since version 1.3. It will be removed in version 1.5. The img_size argument is not required anymore and checks on the input size are run during forward().
  warn_deprecated(argname, msg, warning_category)


nnunet_style (1, 3, 256, 256)


In [7]:
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
)


class NoOpGradScaler:
    def scale(self, loss):
        return loss

    def step(self, optimizer):
        optimizer.step()

    def update(self):
        return None


class _NullAutocast:
    def __enter__(self):
        return None

    def __exit__(self, exc_type, exc, tb):
        return False


def build_grad_scaler():
    if not AMP_ENABLED:
        return NoOpGradScaler()

    amp_module = getattr(torch, "amp", None)
    if amp_module is not None and hasattr(amp_module, "GradScaler"):
        try:
            return amp_module.GradScaler("cuda", enabled=True)
        except TypeError:
            return amp_module.GradScaler(enabled=True)

    cuda_amp = getattr(torch.cuda, "amp", None)
    if cuda_amp is not None and hasattr(cuda_amp, "GradScaler"):
        return cuda_amp.GradScaler(enabled=True)

    return NoOpGradScaler()


def autocast_context():
    amp_module = getattr(torch, "amp", None)
    if amp_module is not None and hasattr(amp_module, "autocast"):
        return amp_module.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED)

    cuda_amp = getattr(torch.cuda, "amp", None)
    if cuda_amp is not None and hasattr(cuda_amp, "autocast"):
        return cuda_amp.autocast(enabled=AMP_ENABLED)

    return _NullAutocast()


def infer_logits(model, images):
    if images.shape[-1] > PATCH_SIZE or images.shape[-2] > PATCH_SIZE:
        return sliding_window_inference(
            images,
            roi_size=(PATCH_SIZE, PATCH_SIZE),
            sw_batch_size=SLIDING_WINDOW_BATCH,
            predictor=model,
            overlap=SLIDING_WINDOW_OVERLAP,
        )
    return model(images)


def compute_metrics_from_arrays(y_true, y_pred):
    flat_true = y_true.reshape(-1)
    flat_pred = y_pred.reshape(-1)
    labels = list(range(NUM_CLASSES))
    return {
        "miou": jaccard_score(flat_true, flat_pred, labels=labels, average="macro", zero_division=0),
        "dice_macro": f1_score(flat_true, flat_pred, labels=labels, average="macro", zero_division=0),
        "precision_macro": precision_score(flat_true, flat_pred, labels=labels, average="macro", zero_division=0),
        "recall_macro": recall_score(flat_true, flat_pred, labels=labels, average="macro", zero_division=0),
        "accuracy": accuracy_score(flat_true, flat_pred),
    }


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    losses = []
    progress = tqdm(loader, leave=False)
    for images, masks in progress:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        progress.set_description(f"train loss {np.mean(losses):.4f}")
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses = []
    all_preds = []
    all_targets = []
    for images, masks in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        with autocast_context():
            logits = infer_logits(model, images)
            loss = criterion(logits, masks)

        preds = torch.argmax(logits, dim=1)
        losses.append(loss.item())
        all_preds.append(preds.cpu().numpy())
        all_targets.append(masks.cpu().numpy())

    y_pred = np.concatenate(all_preds, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    metrics = compute_metrics_from_arrays(y_true, y_pred)
    metrics["loss"] = float(np.mean(losses))
    return metrics


def fit_model(model_name: str):
    set_seed(SEED)
    model = build_model(model_name).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = build_grad_scaler()

    history = []
    best_miou = -1.0
    best_epoch = -1
    wait = 0
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}.pt"

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        eval_metrics = evaluate(model, test_loader)
        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **eval_metrics,
        }
        history.append(row)
        print(
            f"[{model_name}] "
            f"epoch={epoch:03d} "
            f"train_loss={train_loss:.4f} "
            f"test_loss={eval_metrics['loss']:.4f} "
            f"miou={eval_metrics['miou']:.4f} "
            f"dice={eval_metrics['dice_macro']:.4f}"
        )

        if eval_metrics["miou"] > best_miou:
            best_miou = eval_metrics["miou"]
            best_epoch = epoch
            wait = 0
            torch.save(
                {
                    "model_name": model_name,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "metrics": eval_metrics,
                },
                checkpoint_path,
            )
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"[{model_name}] early stopping at epoch {epoch}")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_ROOT / f"{model_name}_history.csv", index=False)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    best_metrics = evaluate(model, test_loader)
    best_metrics["best_epoch"] = best_epoch
    best_metrics["model"] = model_name
    return model, history_df, best_metrics


In [ ]:
trained_models = {}
histories = {}
result_rows = []

for model_name in MODEL_NAMES:
    print("=" * 80)
    model, history_df, best_metrics = fit_model(model_name)
    trained_models[model_name] = model
    histories[model_name] = history_df
    result_rows.append(best_metrics)

results_df = pd.DataFrame(result_rows).sort_values("miou", ascending=False).reset_index(drop=True)
results_df.to_csv(RUN_ROOT / "ablation_results.csv", index=False)
results_df


In [ ]:
plt.figure(figsize=(12, 6))
for model_name, history_df in histories.items():
    plt.plot(history_df["epoch"], history_df["miou"], label=model_name)
plt.xlabel("Epoch")
plt.ylabel("Test mIoU")
plt.title("Ablation mIoU Curves")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
@torch.no_grad()
def predict_single(model, image_np):
    model.eval()
    image_tensor = torch.from_numpy(image_np[None, None]).float().to(DEVICE)
    logits = infer_logits(model, image_tensor)
    pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
    return pred


def plot_prediction_triplet(model_name: str, sample_index: int):
    image = X[sample_index]
    target = Y[sample_index]
    pred = predict_single(trained_models[model_name], image)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.imshow(image, cmap="gray")
    plt.title("Input")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    show_mask(target, title="Ground Truth")

    plt.subplot(1, 3, 3)
    show_mask(pred, title=f"{model_name} Prediction")

    plt.tight_layout()
    plt.show()

    out_dir = PRED_DIR / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"prediction_{sample_index}.npy", pred)


example_model = results_df.loc[0, "model"]
example_index = int(test_idx[0])
print("Best model:", example_model, "example sample:", example_index)
plot_prediction_triplet(example_model, example_index)


In [ ]:
def export_png_for_nnunet(indices, split_name):
    export_root = RUN_ROOT / "nnunetv2_export"
    image_dir = export_root / ("imagesTr" if split_name == "train" else "imagesTs")
    label_dir = export_root / ("labelsTr" if split_name == "train" else "labelsTs")
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for local_id, dataset_idx in enumerate(tqdm(indices, desc=f"Exporting {split_name}")):
        image_uint16 = np.clip(X[dataset_idx] * 65535.0, 0, 65535).astype(np.uint16)
        label_uint8 = Y[dataset_idx].astype(np.uint8)
        stem = f"live_dead_{local_id:04d}"
        Image.fromarray(image_uint16).save(image_dir / f"{stem}_0000.png")
        Image.fromarray(label_uint8).save(label_dir / f"{stem}.png")

    dataset_json = {
        "channel_names": {"0": "mSLIM"},
        "labels": {"background": 0, "live": 1, "dead": 2},
        "numTraining": int(len(train_idx)),
        "file_ending": ".png",
    }
    with open(export_root / "dataset.json", "w") as f:
        json.dump(dataset_json, f, indent=2)

    print("Exported nnUNet-style PNG dataset to:", export_root)
    print("If needed, install with: pip install nnunetv2")
